# HyperElasticity 3D — FEniCS Benchmark

This notebook runs the FEniCS (DOLFINx) HyperElasticity solvers and displays results.

**Prerequisites**: Run inside the devcontainer (or any environment where DOLFINx >= 0.10 is available).

## Problem

Neo-Hookean energy minimisation on a 3D rectangular beam $[0, 0.4] \times [0, 0.1]^2$:

$$J(u) = \int_\Omega W(F(u))\, dx, \quad W(F) = C_1(I_1 - 3 - 2\ln J) + D_1(J - 1)^2$$

where $F = I + \nabla u$ is the deformation gradient and $J = \det F$.
Boundary conditions: homogeneous Dirichlet on $x=0$ (left face), prescribed **rotation** on $x=0.4$ (right face) applied in incremental steps from $0°$ to $360°$.

## Solvers

- `HyperElasticity3D_fenics/solve_HE_custom_jaxversion.py` — **Custom Newton (recommended)**: golden-section line search on $[-0.5, 2]$, GMRES + HYPRE BoomerAMG, near-nullspace (6 rigid-body modes), `--pc_setup_on_ksp_cap` to reuse the AMG preconditioner across Newton steps. Solves all 96 steps.
- `HyperElasticity3D_fenics/solve_HE_snes_newton.py` — **SNES Newton** (`newtonls`): PETSc built-in solver with GMRES + HYPRE AMG defaults. Solves 93/96 steps (fails at extreme deformation near 360°).

Both solvers use the 6 rigid-body near-nullspace modes and GMRES (not CG) since the Jacobian is non-symmetric under large deformation.

> **This notebook demos level 1, 1 step** for a quick run. For the full 96-step benchmark, see the `Full benchmark` cell at the bottom.

In [ ]:
import subprocess, json, os, sys
import pandas as pd
from IPython.display import Markdown, display

# Must be run from the repo root so solver imports resolve correctly
print(f"Working directory: {os.getcwd()}")

## Helper: run a HE solver and collect results

In [ ]:
def run_he_solver(script, nprocs=1, level=1, steps=1, extra_args=None):
    """Run a HyperElasticity solver script and return parsed JSON results.

    Args:
        script:     Path to the solver script relative to repo root.
        nprocs:     Number of MPI processes (1 = serial).
        level:      Mesh refinement level. Level 1 = coarsest (80×2×2 elements).
        steps:      Number of rotation steps to run (each step = 360°/total_steps rotation).
        extra_args: Additional command-line arguments passed directly to the script.
    """
    tmp_json = "/tmp/_he_benchmark_result.json"

    # Build the command: optionally prefix with mpirun for parallel runs
    cmd = []
    if nprocs > 1:
        cmd = ["mpirun", "-n", str(nprocs)]
    cmd += ["python3", script,
            "--level", str(level),
            "--steps", str(steps),
            "--out", tmp_json]
    if extra_args:
        cmd += extra_args

    print(f"Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    # Show stdout so per-step progress is visible
    if result.stdout.strip():
        print(result.stdout)
    if result.returncode != 0:
        print(f"STDERR:\n{result.stderr}", file=sys.stderr)
        return None

    with open(tmp_json) as f:
        data = json.load(f)
    os.remove(tmp_json)
    return data


def display_he_results(data):
    """Format the per-step result list into a readable table."""
    if data is None:
        print("No results (solver failed).")
        return
    print(f"mesh_level={data['mesh_level']}  total_dofs={data['total_dofs']}")
    steps = data["steps"]
    # Select columns common to both Custom and SNES outputs
    rows = []
    for s in steps:
        rows.append({
            "step":    s["step"],
            "angle°":  round(s["angle"] * 180 / 3.141592653589793, 2),
            "time(s)": s["time"],
            "iters":   s["iters"],
            "energy":  s["energy"],
            # 'reason' key only present in SNES output; 'message' in Custom output
            "status":  s.get("message", s.get("reason", "")),
        })
    df = pd.DataFrame(rows)
    display(df.to_markdown(index=False))

## 1. Custom Newton — Serial (recommended)

Golden-section line search, GMRES + HYPRE BoomerAMG with 6 rigid-body near-nullspace modes.
`--pc_setup_on_ksp_cap` reuses the AMG preconditioner when the linear solver converges within the KSP iteration budget, avoiding redundant expensive setup calls.

In [ ]:
custom_serial = run_he_solver(
    "HyperElasticity3D_fenics/solve_HE_custom_jaxversion.py",
    nprocs=1,
    level=1,
    steps=1,   # Single rotation step for quick demo
    extra_args=[
        "--total_steps", "96",    # Total steps for the full 0°→360° rotation (sets step size)
        "--ksp_rtol", "1e-1",     # Loose inner tolerance — sufficient and much faster than tight
        "--ksp_max_it", "30",     # Cap KSP iterations to control cost per Newton step
        "--pc_setup_on_ksp_cap",  # Reuse AMG preconditioner when KSP converges cheaply
    ]
)
display_he_results(custom_serial)

## 2. Custom Newton — Parallel (4 processes)

Same algorithm distributed over 4 MPI ranks. The mesh is decomposed automatically by DOLFINx.

In [ ]:
custom_4proc = run_he_solver(
    "HyperElasticity3D_fenics/solve_HE_custom_jaxversion.py",
    nprocs=4,
    level=1,
    steps=1,
    extra_args=[
        "--total_steps", "96",
        "--ksp_rtol", "1e-1",
        "--ksp_max_it", "30",
        "--pc_setup_on_ksp_cap",
        "--quiet",
    ]
)
display_he_results(custom_4proc)

## 3. SNES Newton — Serial

PETSc built-in `newtonls` solver. Uses GMRES + HYPRE BoomerAMG with default coarsening (no explicit `nodal_coarsen` / `vec_interp_variant`). Near-nullspace is attached to the Hessian matrix.

> **Note**: `vec_interp_variant=3` (AMS-style vector interpolation) leads to a non-symmetric AMG and CG breakdown (`KSP_DIVERGED_BREAKDOWN`) with SNES. GMRES + HYPRE defaults is the correct configuration.

In [ ]:
snes_serial = run_he_solver(
    "HyperElasticity3D_fenics/solve_HE_snes_newton.py",
    nprocs=1,
    level=1,
    steps=1,
    extra_args=[
        "--total_steps", "96",       # Sets the step angle (= 360°/96 per step)
        "--snes_type", "newtonls",   # Line-search Newton
        "--linesearch", "basic",     # Full-step; HE is nearly convex so this works
        "--ksp_type", "gmres",
        "--pc_type", "hypre",
        "--ksp_rtol", "1e-1",
        "--ksp_max_it", "500",
        "--snes_atol", "1e-3",
    ]
)
display_he_results(snes_serial)

## 4. SNES Newton — Parallel (4 processes)

In [ ]:
snes_4proc = run_he_solver(
    "HyperElasticity3D_fenics/solve_HE_snes_newton.py",
    nprocs=4,
    level=1,
    steps=1,
    extra_args=[
        "--total_steps", "96",
        "--snes_type", "newtonls",
        "--linesearch", "basic",
        "--ksp_type", "gmres",
        "--pc_type", "hypre",
        "--ksp_rtol", "1e-1",
        "--ksp_max_it", "500",
        "--snes_atol", "1e-3",
        "--quiet",
    ]
)
display_he_results(snes_4proc)

## 5. Comparison: Custom vs SNES, Serial vs Parallel

In [ ]:
def he_to_df(data, label):
    """Flatten HE step results into a DataFrame with a solver label column."""
    if data is None:
        return pd.DataFrame()
    rows = []
    for s in data["steps"]:
        rows.append({
            "step":    s["step"],
            "time(s)": s["time"],
            "iters":   s["iters"],
            "energy":  s["energy"],
            "solver":  label,
        })
    return pd.DataFrame(rows)

frames = [
    he_to_df(custom_serial, "Custom serial"),
    he_to_df(custom_4proc,  "Custom 4-proc"),
    he_to_df(snes_serial,   "SNES serial"),
    he_to_df(snes_4proc,    "SNES 4-proc"),
]
frames = [f for f in frames if not f.empty]

if frames:
    all_df = pd.concat(frames, ignore_index=True)
    # Pivot so each solver is a column, showing time and iters side by side
    pivot = all_df.pivot_table(
        index=["step"],
        columns="solver",
        values=["time(s)", "iters", "energy"]
    )
    pivot.columns = [f"{c[1]} {c[0]}" for c in pivot.columns]
    display(Markdown(pivot.to_markdown()))

## 6. Full Benchmark (all 96 rotation steps)

To reproduce the full results from `results_HyperElasticity3D.md`, change `steps=1` to `steps=96` in the cells above, or run from the terminal:

**Custom Newton (recommended — all 96/96 steps converge):**
```bash
python3 HyperElasticity3D_fenics/solve_HE_custom_jaxversion.py \
    --level 1 --steps 96 --total_steps 96 \
    --ksp_rtol 1e-1 --ksp_max_it 30 --pc_setup_on_ksp_cap \
    --quiet --out results/he_custom_l1.json
```

**SNES Newton (93/96 steps converge — fails near 360° due to AMG degradation):**
```bash
python3 HyperElasticity3D_fenics/solve_HE_snes_newton.py \
    --level 1 --steps 96 --total_steps 96 \
    --ksp_type gmres --pc_type hypre --ksp_rtol 1e-1 --ksp_max_it 500 --snes_atol 1e-3 \
    --quiet --out results/he_snes_l1.json
```

**Parallel (4 processes):**
```bash
mpirun -n 4 python3 HyperElasticity3D_fenics/solve_HE_custom_jaxversion.py \
    --level 1 --steps 96 --total_steps 96 \
    --ksp_rtol 1e-1 --ksp_max_it 30 --pc_setup_on_ksp_cap \
    --quiet --out results/he_custom_l1_4proc.json
```